In [ ]:
import os, pickle, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.axes as axes
import pprint
from tqdm import tqdm
import multiprocessing
import re
from seaborn import heatmap
from functools import reduce
import seaborn as sns

In [ ]:
import torch
from transformer_lens import HookedTransformer

def load_models(model_name, device='cuda:4', checkpoint=""):
    model = HookedTransformer.from_pretrained_no_processing(
        model_name,
        dtype=torch.bfloat16,
        center_unembed=True,
        center_writing_weights=True,
        #fold_ln=True,
        device=device,
        trust_remote_code=True,
        cache_dir="/mounts/data/proj/olmo_models",
        checkpoint_value=checkpoint,
    )

    return model


# Load the models
model_name = "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:0', checkpoint="main")

In [ ]:
def traverse_directory(root_directory):
    """Traverse directory and collect all .pkl file paths."""
    file_paths = []
    for dirpath, _, filenames in os.walk(root_directory):
        for filename in filenames:
            if filename.endswith(".pkl") and "subgraph" not in filename:  # Adjust based on your file format
                file_paths.append(os.path.join(dirpath, filename))
    return file_paths

def extract_model_and_relation(file_path):
    """Extract model name and relation name from file path."""
    path_parts = file_path.split(os.sep)
    model_name = path_parts[-2]  # Model name is the second last part of the path
    relation_name = os.path.splitext(path_parts[-1])[0]  # File name without extension
    return model_name, relation_name

def read_single_file(file_path):
    """Load data from a single .pkl file and return the model, relation, and data."""
    model_name, relation_name = extract_model_and_relation(file_path)
    with open(file_path, 'rb') as file:
        data_entries = pickle.load(file)
    return model_name, relation_name, data_entries

def read_all_files(file_paths):
    """Reads all .pkl files and organizes data by model and relation sequentially."""
    models_data = {}
    total_files = len(file_paths)
    
    print(f"Total files to process: {total_files}")

    # Sequentially read each file
    for file_path in tqdm(file_paths, total=total_files, desc="Processing files"):
        model_name, relation_name, data_entries = read_single_file(file_path)
        if model_name not in models_data:
            models_data[model_name] = {}
        models_data[model_name][relation_name] = data_entries

    return models_data

In [ ]:
root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

# Step 1: Traverse directory and collect file paths
file_paths = traverse_directory(root_directory)
main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path)]

models_data = read_all_files(main_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")

In [ ]:
sorted_models = sorted(
    models_data.keys(),
    key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if re.search(r'step(\d+)', name) and "main" not in name else float('inf')
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Initialize a dictionary to track switches between layers across models
layer_switches_across_models = {layer_idx: [] for layer_idx in range(33)}

# Loop through models and compare layer X from one model to layer X of the next model
for model_idx in range(len(sorted_models) - 1):  # Exclude the last model since it has no "next" model
    model_name = sorted_models[model_idx]
    next_model_name = sorted_models[model_idx + 1]

    # Get the relevant relations data for the current and next models
    relations_data_cur = models_data[model_name]["country_capital_city"]
    relations_data_next = models_data[next_model_name]["country_capital_city"]

    # Loop through the relations data
    for j in range(len(relations_data_cur)):
        # Extract the top tokens for the current and next models
        resid_top_tokens_cur = np.array(relations_data_cur[j]["resid_top_tokens"])
        resid_top_tokens_next = np.array(relations_data_next[j]["resid_top_tokens"])

        step = 2
        selected_indices = list(range(1, resid_top_tokens_cur.shape[0], step))
        if resid_top_tokens_cur.shape[0] - 1 not in selected_indices:
            selected_indices.append(resid_top_tokens_cur.shape[0] - 1)

        result_cur = resid_top_tokens_cur[selected_indices, :, :]
        result_next = resid_top_tokens_next[selected_indices, :, :]

        # Get the answer token span for the current and next models
        cur_ans_token = [i - 1 for i in relations_data_cur[j]['answer_token_span']]
        next_ans_token = [i - 1 for i in relations_data_next[j]['answer_token_span']]

        # Loop through layers and compare between models
        for layer_idx in range(result_cur.shape[0]):  # Compare corresponding layers across models
            # Get tokens from the current model's layer and the next model's corresponding layer
            cur_tokens_at_layer = [result_cur[layer_idx, cur_ans_token][0][0]]
            next_tokens_at_layer = [result_next[layer_idx, next_ans_token][0][0]]

            # Convert all tokens to string using model.to_str_tokens() for both current and next models
            cur_tokens_str = [model.to_string([token]) for token in cur_tokens_at_layer]
            next_tokens_str = [model.to_string([token]) for token in next_tokens_at_layer]

            # Compare the tokens between the corresponding layers
            for cur_token, next_token in zip(cur_tokens_str, next_tokens_str):
                if cur_token != next_token:  # If the token switches
                    layer_switches_across_models[layer_idx].append((cur_token, next_token))  # Track the token pair (switch) for the current layer

In [ ]:
layer_switches_across_models

In [ ]:
import matplotlib.pyplot as plt

# Count the number of switches for each layer
layer_switch_counts = {layer_idx: len(switches) for layer_idx, switches in layer_switches_across_models.items()}

# Prepare data for plotting
layers = list(layer_switch_counts.keys())
switch_counts = list(layer_switch_counts.values())

# Create the plot
plt.figure(figsize=(10, 6))
plt.bar(layers, switch_counts, width=0.6)

# Add labels and title
plt.xlabel("Layer Index", fontsize=14)
plt.ylabel("Number of Token Switches", fontsize=14)
plt.title("Token Switches Across Models by Layer", fontsize=16)
plt.xticks(layers)  # Ensure all layers are labeled
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data for heatmaps
for layer_idx, switches in layer_switches_across_models.items():
    # Create a counter to tally switches
    switch_counter = Counter(switches)
    unique_tokens = list(set([token for pair in switches for token in pair]))  # Unique tokens involved in switches
    token_to_idx = {token: i for i, token in enumerate(unique_tokens)}  # Map tokens to indices

    # Initialize the heatmap matrix
    heatmap_matrix = np.zeros((len(unique_tokens), len(unique_tokens)))

    # Populate the heatmap matrix with counts of switches
    for (from_token, to_token), count in switch_counter.items():
        heatmap_matrix[token_to_idx[from_token], token_to_idx[to_token]] = count

    # Plot the heatmap for the current layer
    plt.figure(figsize=(8, 8))
    plt.imshow(heatmap_matrix, cmap="viridis", interpolation="nearest")
    plt.colorbar(label="Number of Switches")

    # Add labels and title
    plt.xticks(ticks=range(len(unique_tokens)), labels=unique_tokens, rotation=90, fontsize=10)
    plt.yticks(ticks=range(len(unique_tokens)), labels=unique_tokens, fontsize=10)
    plt.title(f"Token Switches in Layer {layer_idx} Across Models", fontsize=14)
    plt.xlabel("Switched To (Target Tokens)", fontsize=12)
    plt.ylabel("Switched From (Source Tokens)", fontsize=12)
    plt.tight_layout()

    # Show the plot
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Define a threshold for minimum switch count to include in the plot
threshold = 5  # Only include switches occurring at least 5 times

for layer_idx in range(32):  # Assuming 32 layers
    plt.figure(figsize=(20, 10))
    
    # Access the layer data for the switches across models
    layer_data = layer_switches_across_models.get(layer_idx, [])
    
    # Check if there are switches in this layer across models
    if layer_data:
        # Count the frequency of each switch pair in this layer
        switch_pair_counts = Counter(layer_data)

        # Filter the switches that have a count greater than or equal to the threshold
        filtered_switch_pairs = {
            pair: count for pair, count in switch_pair_counts.items() if count >= threshold
        }

        # If there are no pairs with count greater than or equal to the threshold, skip this layer
        if filtered_switch_pairs:
            # Prepare labels and values for plotting
            switch_labels = [f"{pair[0]} -> {pair[1]}" for pair in filtered_switch_pairs.keys()]
            switch_values = list(filtered_switch_pairs.values())

            # Create the horizontal bar plot
            plt.barh(switch_labels, switch_values, color='skyblue')
            
            # Add labels and title
            plt.xlabel('Frequency of Switch', fontsize=14)
            plt.ylabel('Token Pair (Switches)', fontsize=14)
            plt.title(f'Token Switches (Count >= {threshold}) for Layer {layer_idx} Across Models', fontsize=16)
            plt.tight_layout()
            plt.show()
        else:
            # No switches meeting the threshold for this layer
            print(f"No switches meeting the threshold for Layer {layer_idx} across models.")
            plt.close()
    else:
        # No switches in this layer across models
        print(f"No switches to plot for Layer {layer_idx} across models.")
        plt.close()


In [ ]:
sorted_models

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Initialize a list to keep track of switches between layers for each model
layer_switches_within_model = {model_name: {layer_idx: [] for layer_idx in range(32)} for model_name in sorted_models}

for model_name in sorted_models:
    #if model_name == 'main':
    #    continue

    # Get the relevant relations data for the current model
    relations_data = models_data[model_name]["country_capital_city"]

    # Loop through the relations data
    for j in range(len(relations_data)):
        # Extract the top tokens for the current model
        resid_top_tokens_cur = np.array(relations_data[j]["resid_top_tokens"])#[1::2]  # Assuming resid_top_tokens is a numpy array

        step = 2
        selected_indices = list(range(1, resid_top_tokens_cur.shape[0], step))
        if resid_top_tokens_cur.shape[0] - 1 not in selected_indices:
            selected_indices.append(resid_top_tokens_cur.shape[0] - 1)

        result = resid_top_tokens_cur[selected_indices, :, :]

        # Get the answer token span for the current model
        cur_ans_token = [i - 1 for i in relations_data[j]['answer_token_span']]

        # Loop through layers and compare consecutive layers
        for layer_idx in range(result.shape[0] - 1):  # Iterate through layers (first dimension: 65 layers, compare layer i vs. layer i+1)
            next_layer_idx = layer_idx + 1

            # Get tokens from the current layer and the next layer
            cur_tokens_at_layer = [result[layer_idx, cur_ans_token][0][0]]
            next_tokens_at_layer = [result[next_layer_idx, cur_ans_token][0][0]]

            # Convert all tokens to string using model.to_str_tokens() for both current and next layers
            cur_tokens_str = [model.to_string([token]) for token in cur_tokens_at_layer]
            next_tokens_str = [model.to_string([token]) for token in next_tokens_at_layer]

            # Compare the tokens between the consecutive layers
            for cur_token, next_token in zip(cur_tokens_str, next_tokens_str):
                if cur_token != next_token:  # If the token switches
                    layer_switches_within_model[model_name][layer_idx].append((cur_token, next_token))  # Track the token pair (switch) for the current layer

            # Print the layer and all tokens for debugging purposes
            #print(f"Model: {model_name}, Relation: {relations_data[j]['sentence']}, Layer: {layer_idx} vs {next_layer_idx}")
            #print(f"Current layer tokens at layer {layer_idx}: {cur_tokens_str}")
            #print(f"Next layer tokens at layer {next_layer_idx}: {next_tokens_str}")
            #print("-" * 50)


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Specify the model you want to plot
selected_model = "main"#'main'  # Replace 'main' with the desired model name

# Define a threshold for minimum switch count to include in the plot
threshold = 0  # Adjust as needed

for layer_idx in range(32):  # Assuming 32 layers
    plt.figure(figsize=(20, 10))
    
    # Access the layer data for the selected model
    layer_data = layer_switches_within_model.get(selected_model, {})
    
    # Check if there are switches in this layer for the selected model
    if layer_data.get(layer_idx):
        # Count the frequency of each switch pair in this layer
        switch_pair_counts = Counter(layer_data[layer_idx])

        # Filter the switches that have a count greater than the threshold
        filtered_switch_pairs = {
            pair: count for pair, count in switch_pair_counts.items() if count > threshold
        }

        # If there are no pairs with count greater than the threshold, skip this layer
        if filtered_switch_pairs:
            # Prepare labels and values for plotting
            switch_labels = [f"{pair[0]} -> {pair[1]}" for pair in filtered_switch_pairs.keys()]
            switch_values = list(filtered_switch_pairs.values())

            # Create the horizontal bar plot
            plt.barh(switch_labels, switch_values, color='skyblue')
            
            plt.xlabel('Frequency of Switch')
            plt.ylabel('Token Pair (Switches)')
            plt.title(f'Token Switches (Count > {threshold}) for Layer {layer_idx} in Model "{selected_model}"')
            plt.tight_layout()
            plt.show()
    else:
        # No switches in this layer for the selected model
        print(f"No switches to plot for Layer {layer_idx} in Model '{selected_model}'.")
        plt.close()


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Specify the model you want to plot
selected_model = "step5000-tokens20B"#'main'  # Replace 'main' with the desired model name

# Define a threshold for minimum switch count to include in the plot
threshold = 0  # Adjust as needed

for layer_idx in range(32):  # Assuming 32 layers
    plt.figure(figsize=(20, 10))
    
    # Access the layer data for the selected model
    layer_data = layer_switches_within_model.get(selected_model, {})
    
    # Check if there are switches in this layer for the selected model
    if layer_data.get(layer_idx):
        # Count the frequency of each switch pair in this layer
        switch_pair_counts = Counter(layer_data[layer_idx])

        # Filter the switches that have a count greater than the threshold
        filtered_switch_pairs = {
            pair: count for pair, count in switch_pair_counts.items() if count > threshold
        }

        # If there are no pairs with count greater than the threshold, skip this layer
        if filtered_switch_pairs:
            # Prepare labels and values for plotting
            switch_labels = [f"{pair[0]} -> {pair[1]}" for pair in filtered_switch_pairs.keys()]
            switch_values = list(filtered_switch_pairs.values())

            # Create the horizontal bar plot
            plt.barh(switch_labels, switch_values, color='skyblue')
            
            plt.xlabel('Frequency of Switch')
            plt.ylabel('Token Pair (Switches)')
            plt.title(f'Token Switches (Count > {threshold}) for Layer {layer_idx} in Model "{selected_model}"')
            plt.tight_layout()
            plt.show()
    else:
        # No switches in this layer for the selected model
        print(f"No switches to plot for Layer {layer_idx} in Model '{selected_model}'.")
        plt.close()
